In [1]:
import pandas as pd
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
import torch
import os

In [2]:

path = "../data/MOT17/"
train_path = path + "train/MOT17-02-FRCNN/"

# Coloums
# frame_number, identity_number, bbox_left, bbox_top, bbox_width, bbox_height, confidence_score, class, visibility

columns = ["frame_number", "identity_number", "bbox_left", "bbox_top", "bbox_width", "bbox_height", "confidence_score", "class", "visibility"]
gt_train = pd.read_csv(train_path + "gt/gt.txt", sep=",", header=None, names=columns)


In [3]:
gt_train

,frame_number,identity_number,bbox_left,bbox_top,bbox_width,bbox_height,confidence_score,class,visibility
0,1,1,912,484,97,109,0,7,1.0
1,2,1,912,484,97,109,0,7,1.0
2,3,1,912,484,97,109,0,7,1.0
3,4,1,912,484,97,109,0,7,1.0
4,5,1,912,484,97,109,0,7,1.0
...,...,...,...,...,...,...,...,...,...
29998,593,80,1043,445,32,97,1,1,0.0
29999,594,80,1043,445,32,97,1,1,0.0
30000,600,81,1007,451,24,69,1,1,0.0
30001,600,82,987,473,21,43,1,1,0.0


In [ ]:
filtered_gt = gt_train.query("confidence_score != 0")
filtered_gt

,frame_number,identity_number,bbox_left,bbox_top,bbox_width,bbox_height,confidence_score,class,visibility
600,1,2,1338,418,167,379,1,1,1.0
601,2,2,1342,417,168,380,1,1,1.0
602,3,2,1346,417,170,380,1,1,1.0
603,4,2,1351,417,171,381,1,1,1.0
604,5,2,1355,417,173,381,1,1,1.0
...,...,...,...,...,...,...,...,...,...
29998,593,80,1043,445,32,97,1,1,0.0
29999,594,80,1043,445,32,97,1,1,0.0
30000,600,81,1007,451,24,69,1,1,0.0
30001,600,82,987,473,21,43,1,1,0.0


In [ ]:
filtered_gt["bbox"] = filtered_gt[["bbox_left", "bbox_top", "bbox_width", "bbox_height"]].values.tolist()
filtered_gt = filtered_gt[["frame_number", "bbox", "class", "visibility"]]
filtered_gt

/tmp/ipykernel_3361226/511642379.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_gt_train["bbox"] = filtered_gt_train[["bbox_left", "bbox_top", "bbox_width", "bbox_height"]].values.tolist()


,frame_number,bbox,class,visibility
600,1,"[1338, 418, 167, 379]",1,1.0
601,2,"[1342, 417, 168, 380]",1,1.0
602,3,"[1346, 417, 170, 380]",1,1.0
603,4,"[1351, 417, 171, 381]",1,1.0
604,5,"[1355, 417, 173, 381]",1,1.0
...,...,...,...,...
29998,593,"[1043, 445, 32, 97]",1,0.0
29999,594,"[1043, 445, 32, 97]",1,0.0
30000,600,"[1007, 451, 24, 69]",1,0.0
30001,600,"[987, 473, 21, 43]",1,0.0


In [ ]:
combined = filtered_gt.groupby("frame_number").agg({"bbox": list, "class": list, "visibility": list}).reset_index()
combined

,frame_number,bbox,class,visibility
0,1,"[[1338, 418, 167, 379], [586, 447, 85, 263], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51351, 0.94595, 1.0, 1.0, 0.98687..."
1,2,"[[1342, 417, 168, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.5163, 0.94643, 1.0, 1.0, 0.98666,..."
2,3,"[[1346, 417, 170, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51366, 0.94643, 1.0, 1.0, 0.98666..."
3,4,"[[1351, 417, 171, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51099, 0.94643, 1.0, 1.0, 0.98645..."
4,5,"[[1355, 417, 173, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.50829, 0.94643, 1.0, 1.0, 0.98645..."
...,...,...,...,...
595,596,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.085047, 0.13523, 0.2027, 1.0, 0.0..."
596,597,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56505, 0.14369, 0.22491, 1.0, 0.0..."
597,598,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.55093, 0.14098, 0.25149, 1.0, 0.0..."
598,599,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56644, 0.14927, 0.27751, 1.0, 0.0..."


In [7]:
image_paths = []
for frame_number in combined["frame_number"]:
    image_file = f"{frame_number:06d}.jpg"
    image_path = os.path.join(train_path, "img1", image_file)
    image_paths.append(image_path)

combined["image_path"] = image_paths
combined

,frame_number,bbox,class,visibility,image_path
0,1,"[[1338, 418, 167, 379], [586, 447, 85, 263], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51351, 0.94595, 1.0, 1.0, 0.98687...",../data/MOT17/train/MOT17-02-FRCNN/img1/000001...
1,2,"[[1342, 417, 168, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.5163, 0.94643, 1.0, 1.0, 0.98666,...",../data/MOT17/train/MOT17-02-FRCNN/img1/000002...
2,3,"[[1346, 417, 170, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51366, 0.94643, 1.0, 1.0, 0.98666...",../data/MOT17/train/MOT17-02-FRCNN/img1/000003...
3,4,"[[1351, 417, 171, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51099, 0.94643, 1.0, 1.0, 0.98645...",../data/MOT17/train/MOT17-02-FRCNN/img1/000004...
4,5,"[[1355, 417, 173, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.50829, 0.94643, 1.0, 1.0, 0.98645...",../data/MOT17/train/MOT17-02-FRCNN/img1/000005...
...,...,...,...,...,...
595,596,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.085047, 0.13523, 0.2027, 1.0, 0.0...",../data/MOT17/train/MOT17-02-FRCNN/img1/000596...
596,597,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56505, 0.14369, 0.22491, 1.0, 0.0...",../data/MOT17/train/MOT17-02-FRCNN/img1/000597...
597,598,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.55093, 0.14098, 0.25149, 1.0, 0.0...",../data/MOT17/train/MOT17-02-FRCNN/img1/000598...
598,599,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56644, 0.14927, 0.27751, 1.0, 0.0...",../data/MOT17/train/MOT17-02-FRCNN/img1/000599...


In [41]:
from typing import Callable
from PIL import Image

import torchvision.transforms.functional as TF

class VODDataset(Dataset):
    # sequence_dirs: list of directories, each containing frames of a video sequence
    # n_frames: number of consecutive frames to include in each clip
    # get_item should return a dict with keys "pixel_values" (list of tensors) and "labels" (list of dicts)
    # "pixel_values" should be a list of tensors of shape (Frames, Channels, Height, Width)
    # "labels" should be a list of dicts, one per frame, each containing the COCO-style annotations for that frame
    def __init__(
        self,
        sequence_dirs: list[str],
        n_frames: int = 4,
        # sampling_strategy: str = "consecutive",  # or "random"
        transform: Callable | None = None,
    ):
        self.n_frames = n_frames
        self.transform = transform
        self.sequence_dirs = sequence_dirs
        self.columns = [
            "frame_number",
            "identity_number",
            "bbox_left",
            "bbox_top",
            "bbox_width",
            "bbox_height",
            "confidence_score",
            "class",
            "visibility",
        ]
        self.clips: list[tuple[str, list[str]]] = []
        self.gt_annotations: dict[str, pd.DataFrame] = {}

        
        """
        Sequence to sequence predictions: for each sequence, we can create multiple clips by sliding a window of size n_frames across the frames. For example, if a sequence has 10 frames and n_frames=4, we can create the following clips:
            - Clip 1: frames 1-4
            - Clip 2: frames 2-5
            - Clip 3: frames 3-6
        They also apply normal image augmentations: random horizontal flip and random resizing
        """
        """
        sequence_dir
        ├── img1/
        │   ├── 00001.jpg
        │   ├── 00002.jpg
        │   └── ...
        ├── gt/
        │   └── gt.txt
        └── sequence.ini

        """
        for seq_dir in sequence_dirs:
            img_dir = os.path.join(seq_dir, "img1")
            frame_files = sorted(
                [
                    f
                    for f in os.listdir(img_dir)
                    if f.endswith((".jpg", ".png", ".jpeg"))
                ]
            )
            for i in range(len(frame_files) - n_frames + 1):
                clip_frames = frame_files[i : i + n_frames]
                self.clips.append((seq_dir, clip_frames))
            gt_path = os.path.join(seq_dir, "gt", "gt.txt")
            self.prepare_dataframe(seq_dir, gt_path)

    def prepare_dataframe(self, seq_dir: str, gt_path: str):
        gt = pd.read_csv(gt_path, sep=",", header=None, names=self.columns)
        filtered_gt = gt.query("confidence_score != 0")
        filtered_gt["bbox"] = filtered_gt[["bbox_left", "bbox_top", "bbox_width", "bbox_height"]].values.tolist()
        filtered_gt = filtered_gt[["frame_number", "bbox", "class", "visibility"]]
        combined = filtered_gt.groupby("frame_number").agg({"bbox": list, "class": list, "visibility": list}).reset_index()
        # class to category
        combined.rename(columns={"class": "category"})
        self.gt_annotations[seq_dir] = combined

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, index):
        seq_dir: str
        clip_frames: list[str]
        seq_dir, clip_frames = self.clips[index]
        pixel_values = []
        labels = []
        for frame_file in clip_frames:
            image, annotation = self.load_image(seq_dir, frame_file)
            if self.transform is not None:
                image, annotation = self.transform(image, annotation)
            pixel_values.append(image)
            labels.append({"objects": annotation})
        # pixel_values = torch.stack(pixel_values)  # (Frames, Channels, Height, Width)
        return {"pixel_values": pixel_values, "labels": labels}

    def load_image(self, seq_dir: str, frame_file: str) -> tuple[torch.Tensor, dict]:
        frame_path = os.path.join(seq_dir, "img1", frame_file)
        img = Image.open(frame_path).convert("RGB")
        tensor = TF.to_tensor(img)  # (3, H, W) float32

        frame_number = int(os.path.splitext(frame_file)[0])
        gt = self.gt_annotations[seq_dir]

        row = gt[gt["frame_number"] == frame_number].to_dict()
        return tensor, row


In [ ]:
dataset = VODDataset(sequence_dirs=[train_path], n_frames=4)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
for batch in dataloader:
    print(batch)
    break


/tmp/ipykernel_3361226/2922671716.py:73: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_gt["bbox"] = filtered_gt[["bbox_left", "bbox_top", "bbox_width", "bbox_height"]].values.tolist()


TypeError: list indices must be integers or slices, not list